# fastbook Chapter 2 — From Model to Production

Exercises worked through alongside chapter 2 of the
[fast.ai book](https://github.com/fastai/fastbook) by Jeremy Howard & Sylvain Gugger.

> These notebooks are derived from the fast.ai book and are licensed
> [CC BY-NC 4.0](https://creativecommons.org/licenses/by-nc/4.0/), **not** the MIT license
> that covers the rest of this repository.

Where chapter 1 trained models, this chapter is about everything around them: gathering
your own data, checking it is any good, fixing it when it is not, and putting the result
somewhere a person can actually use. The running example is a three-way bear classifier —
grizzly, black, teddy.

In [ ]:
! [ -e /content ] && pip install -Uqq fastbook
import fastbook
fastbook.setup_book()

In [ ]:
from fastbook import *
from fastai.vision.widgets import *

## Gathering the data

There is no ready-made bear dataset, so the images come from web search. As in the bird
exercise, the search helper retries with backoff because the endpoint rate-limits when
pulling images in bulk.

In [ ]:
!pip install -Uq ddgs
from ddgs import DDGS
from fastcore.all import L
import time, random

def search_images(term, max_images=30, retries=5):
  print(f"Searching for '{term}'")
  for attempt in range(retries):
    try:
      with DDGS() as ddgs:
        results = ddgs.images(query=term, max_results=max_images)
        return L([r['image'] for r in results])
    except Exception as e:
      wait = 15 * (attempt + 1) + random.randint(5, 15)
      print(f"  {e.__class__.__name__}: {e}, waiting {wait}s (attempt {attempt+1}/{retries})")
      time.sleep(wait)
  print(f"  Failed after {retries} retries, returning empty list")
  return L([])

In [ ]:
search_images

In [ ]:
urls = search_images('grizzly bear', max_images=1)
urls[0]

In [ ]:
dest = 'images/grizzly.jpg'
download_url(urls[0], dest)

In [ ]:
im = Image.open(dest)
im.to_thumb(128,128)

In [ ]:
bear_types = 'grizzly','black','teddy'
path = Path('bears')

In [ ]:
import time, random

if not path.exists():
  path.mkdir()
  for o in bear_types:
    dest = (path/o)
    dest.mkdir(exist_ok=True)
    download_images(dest, urls=search_images(f'{o} bear', max_images=50))
    time.sleep(15 + random.randint(5, 15))


## Checking what was downloaded

A web scrape always contains broken files. List what arrived, verify it, and unlink
whatever fails to open.

In [ ]:
fns = get_image_files(path)
fns

In [ ]:
failed = verify_images(fns)
failed

In [ ]:
failed.map(Path.unlink);

## The DataBlock API

`DataBlock` is a *description* of the pipeline — what the inputs and targets are, where
items come from, how to label them, how to split them — and is only executed when
`dataloaders` is called on a path. That separation is what makes the next section possible.

In [ ]:
bears = DataBlock(
    blocks=(ImageBlock, CategoryBlock),
    get_items=get_image_files,
    splitter=RandomSplitter(valid_pct=0.2, seed=42),
    get_y=parent_label,
    item_tfms=Resize(128)    
)

In [ ]:
dls =  bears.dataloaders(path)

In [ ]:
dls.valid.show_batch(max_n=4, nrows=1)

## Choosing a resize strategy

Every image has to reach a common size before it can be batched, and there is more than one
way to get there. `bears.new(...)` rebuilds the pipeline with one setting changed so the
options can be compared side by side:

- **Crop** (the default) — keeps the scale, discards the edges.
- **Squish** — keeps everything, distorts the aspect ratio.
- **Pad** — keeps everything undistorted, wastes pixels on padding.
- **RandomResizedCrop** — a different crop every epoch, which doubles as augmentation.

The final configuration uses `RandomResizedCrop` at 224 px plus `aug_transforms`, so the
model sees a slightly different version of each image every time.

In [ ]:
bears = bears.new(item_tfms=Resize(128, ResizeMethod.Squish))
dls = bears.dataloaders(path)
dls.valid.show_batch(max_n=4, nrows=1)

In [ ]:
bears = bears.new(item_tfms=Resize(128, ResizeMethod.Pad, pad_mode='zeros'))
dls = bears.dataloaders(path)
dls.valid.show_batch(max_n=4, nrows=1)

In [ ]:
bears = bears.new(item_tfms=RandomResizedCrop(128, min_scale=0.3))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=4, nrows=1, unique=True)

In [ ]:
bears = bears.new(item_tfms=Resize(128), batch_tfms=aug_transforms(mult=2))
dls = bears.dataloaders(path)
dls.train.show_batch(max_n=8, nrows=2, unique=True)

In [ ]:
bears = bears.new(
    item_tfms=RandomResizedCrop(224, min_scale=0.5),
    batch_tfms=aug_transforms())
dls = bears.dataloaders(path)

## Train, then clean

The order here is the interesting part, and it is counter-intuitive: train *first*, then
use the model to find the bad data. The confusion matrix shows which categories get mixed
up, and `plot_top_losses` surfaces the images the model was most confidently wrong about —
which is exactly where mislabelled and junk images tend to sit.

`ImageClassifierCleaner` then makes that fixable: it displays the worst offenders sorted by
loss and lets you mark each one for deletion or relabelling. It only records the choices —
the two cells after it are what actually move and delete the files.

In [ ]:
learn = vision_learner(dls, resnet18, metrics=error_rate)
learn.fine_tune(4)

In [ ]:
interp = ClassificationInterpretation.from_learner(learn)
interp.plot_confusion_matrix()

In [ ]:
interp.plot_top_losses(5, nrows=1)

In [ ]:
cleaner = ImageClassifierCleaner(learn)
cleaner

In [ ]:
for idx in cleaner.delete(): cleaner.fns[idx].unlink()

In [ ]:
for idx,cat in cleaner.change(): shutil.move(str(cleaner.fns[idx]), path/cat)

## Export and inference

`learn.export()` saves the model with its inference pipeline attached. Loading it back with
`load_learner` gives an object that predicts on a single image without any of the training
setup — this is what gets deployed.

In [ ]:
learn.export()

In [ ]:
path = Path()
path.ls(file_exts='.pkl')

In [ ]:
learn_inf = load_learner(path/'export.pkl')

In [ ]:
learn_inf.predict('images/grizzly.jpg')

In [ ]:
learn_inf.dls.vocab

## A notebook app with ipywidgets

Assembling a minimal interface from widgets: an upload button, a place to show the image,
a classify button, and a label for the prediction. The final `VBox` stacks them into
something usable.

(The `gradio.ipynb` exercise elsewhere in this repo does the same job with a tool built for
it, and the result runs outside the notebook.)

In [ ]:
btn_upload = widgets.FileUpload()
btn_upload

In [ ]:
img = PILImage.create(btn_upload.data[-1])

In [ ]:
out_pl = widgets.Output()
out_pl.clear_output()
with out_pl: display(img.to_thumb(128,128))
out_pl

In [ ]:
pred,pred_idx,probs = learn_inf.predict(img)

In [ ]:
lbl_pred = widgets.Label()
lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'
lbl_pred

In [ ]:
btn_run = widgets.Button(description='Classify')
btn_run

In [ ]:
def on_click_classify(change):
    img = PILImage.create(btn_upload.data[-1])
    out_pl.clear_output()
    with out_pl: display(img.to_thumb(128,128))
    pred,pred_idx,probs = learn_inf.predict(img)
    lbl_pred.value = f'Prediction: {pred}; Probability: {probs[pred_idx]:.04f}'

btn_run.on_click(on_click_classify)

In [ ]:
btn_upload = widgets.FileUpload()

In [ ]:
VBox([widgets.Label('Select your bear!'), 
      btn_upload, btn_run, out_pl, lbl_pred])